# 08 - Hypothesis-informed tests

**Purpose**: run the canonical analysis pipeline (notebooks 01-07) in a
way that explicitly leverages two sources of prior knowledge and
tests their adequacy:

1. **Literature-derived region prior**
   (``mousedb.region_priors.SKILLED_REACHING``). The field already
   has strong hypotheses about which descending regions matter for
   reaching. We test whether adding lower-priority regions gives
   extra predictive power beyond the prior.
2. **eLife groupings**. The grouping lumps atomic regions into
   functional families. If a group's subregions have opposite
   functional roles, the grouping hides variance. We drill down
   per group to test this.

Sections:

1. Grouped vs ungrouped connectivity PCA side-by-side.
2. Per-group drill-down: within each eLife group, PCA on its
   atomic-region constituents.
3. Nested LMM comparison: does adding prior-ranked regions
   improve fit over phase-only?
4. Prior-weighted PCA: does weighting features by predicted
   importance change what the decomposition sees?

Supports ``USE_SYNTHETIC=True`` for pipeline validation at
realistic N (see notebook 07).


In [ ]:
# parameters
USE_SYNTHETIC = False
SYNTHETIC_N = 30
SYNTHETIC_SEED = 42
TOP_K_PRIORS = 5          # How many top-priority regions to add in the nested-LMM step
TARGET_FEATURE = 'max_extent_mm'
PRIOR_DECAY = 0.1         # Exponential-decay rate for prior weighting; 0 = uniform
FIGSIZE_VAR = (10, 5)
FIGSIZE_DRILL = (16, 8)
FIGSIZE_WEIGHTED = (14, 6)


In [ ]:
import numpy as np                                                                          # numerical arrays
import pandas as pd                                                                          # dataframes
import matplotlib.pyplot as plt                                                              # plotting
import seaborn as sns                                                                        # heatmaps
from sklearn.preprocessing import StandardScaler                                             # z-scorer (used twice in this notebook for PCA prep)
from sklearn.decomposition import PCA                                                        # principal component analysis

from endpoint_ck_analysis import SKILLED_REACHING, ordered_hemisphere_columns                # canonical region prior + helper for priority-ordered column lists
from endpoint_ck_analysis.config import CACHE_DIR, EXAMPLE_OUTPUT_DIR, ANALYZABLE_PHASES     # cache dir + output dir + analyzable phase set
from endpoint_ck_analysis.data_loader import load_all                                        # one-shot loader (supports synthetic mode via flags)
from endpoint_ck_analysis.helpers.hierarchical import (                                      # tools for grouped-vs-ungrouped analysis
    build_group_region_map, drill_down_pca, grouped_vs_ungrouped_summary,                    # group->region map + per-group PCA + side-by-side variance summary
)
from endpoint_ck_analysis.helpers.dimreduce import (                                         # tools for prior-weighted decomposition
    priority_weights_from_prior, apply_feature_weights,                                       # build per-feature weights from a region prior + apply them to an X matrix
)
from endpoint_ck_analysis.helpers.models import compare_nested_lmms                          # nested LMM model-comparison helper (AIC/BIC/LRT)
from endpoint_ck_analysis.helpers.plotting import stamp_version                              # figure version footer

EXAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)                                        # ensure output folder exists
data = load_all(                                                                              # data load with synthetic flags wired through
    use_synthetic=USE_SYNTHETIC,
    synthetic_n=SYNTHETIC_N,
    synthetic_seed=SYNTHETIC_SEED,
    use_cache=not USE_SYNTHETIC,                                                             # synthetic always rebuilds (cache reflects last real load)
    write_cache=not USE_SYNTHETIC,                                                           # don't overwrite real-data cache with synthetic
    verbose=False,
)
print(f"Running on {'SYNTHETIC' if USE_SYNTHETIC else 'REAL'} data. "
      f"N={len(data.matched_subjects)} subjects.")


## 1. Grouped vs ungrouped PCA

Run connectivity PCA at both the eLife-group level
(``ACDGdf_wide``, ~80 columns) and the atomic-region level
(``ACDUdf_wide``, ~700 columns). Different variance-explained
profiles imply the grouping is hiding or creating structure.


In [ ]:
summary = grouped_vs_ungrouped_summary(                                                      # helper runs PCA on both matrices and returns a long-format summary
    data.ACDGdf_wide.fillna(0), data.ACDUdf_wide.fillna(0), n_components=5,                  # eLife-grouped matrix vs atomic-region matrix; n_components=5 is enough to show the scree shape
)
print(summary)                                                                                # tabular printout of variance per PC at both levels

fig, ax = plt.subplots(figsize=FIGSIZE_VAR)
for level, grp in summary.groupby('level'):                                                  # one line per level (grouped, ungrouped)
    ax.plot(grp['component'], grp['variance_explained'], marker='o', label=level)            # standard scree-style line plot
ax.set_ylabel('Variance explained')
ax.set_xlabel('Component')
ax.set_title('Connectivity PCA: grouped (eLife) vs ungrouped (atomic regions)')
ax.legend()
ax.grid(alpha=0.3)                                                                            # faint grid for easier reading
stamp_version(fig, label='08 grouped vs ungrouped')
plt.tight_layout()
plt.savefig(EXAMPLE_OUTPUT_DIR / '08_grouped_vs_ungrouped.png', dpi=150, bbox_inches='tight')
plt.show()


## 2. Per-group drill-down

For each eLife group, run a mini-PCA on its atomic-region
columns. A group whose PC1 captures most of its variance is a
coherent bundle; a group where variance is spread across many
PCs has subregional heterogeneity the grouping is hiding.


In [ ]:
group_region_map = build_group_region_map(data.counts_groupeddf)                                # build {group_name: [atomic_region_columns]} map from the long-format grouped counts table
drill_rows = []                                                                                  # accumulator: one dict per group with that group's PCA result
for group in data.FCDGdf_wide.columns.map(lambda c: c.rsplit('_', 1)[0]).unique():               # iterate unique group names; column names are 'GROUP_left'/'GROUP_right'/'GROUP_both' so rsplit('_',1)[0] strips the hemisphere suffix
    result = drill_down_pca(data.ACDUdf_wide.fillna(0), group, group_region_map, n_components=3) # helper: subset atomic columns belonging to this group and run a mini-PCA on them
    if result is None:                                                                            # helper returns None when group has fewer atomic regions than samples (PCA would be ill-posed)
        continue                                                                                  # skip this group; loop continues with the next
    drill_rows.append({                                                                           # build a flat record for the summary dataframe
        'group': group,                                                                           # group name (eLife label)
        'n_atomic': result.n_atomic_regions,                                                      # how many atomic regions fed into this group's mini-PCA
        'PC1_var': float(result.explained_variance_ratio[0]) if len(result.explained_variance_ratio) else np.nan,    # PC1 share of within-group variance; np.nan if no PCs returned
        'PC2_var': float(result.explained_variance_ratio[1]) if len(result.explained_variance_ratio) > 1 else np.nan,  # PC2 share; np.nan if fewer than 2 PCs
        'PC3_var': float(result.explained_variance_ratio[2]) if len(result.explained_variance_ratio) > 2 else np.nan,  # PC3 share; np.nan if fewer than 3 PCs
    })

drill_df = pd.DataFrame(drill_rows).sort_values('n_atomic', ascending=False)                     # convert list-of-dicts to dataframe; sort biggest groups first so the bar chart reads left-to-right by size
print(drill_df.to_string(index=False))                                                            # full table to stdout; to_string(index=False) suppresses the row-number column

# Plot: stacked bar of PC1/PC2/PC3 variance per group
fig, ax = plt.subplots(figsize=FIGSIZE_DRILL)                                                    # figsize from parameters cell so users can resize without editing here
x = range(len(drill_df))                                                                         # x positions: one tick per group
bottom = np.zeros(len(drill_df))                                                                 # running bottom for stacking; starts at zero for PC1
for pc_col, color, label in [('PC1_var', 'steelblue', 'PC1'),                                    # iterate the three PCs; each adds another stacked layer
                             ('PC2_var', 'coral', 'PC2'),
                             ('PC3_var', 'seagreen', 'PC3')]:
    vals = drill_df[pc_col].fillna(0).values                                                     # variance shares; fillna(0) because groups with <3 atomic regions have NaN for higher PCs
    ax.bar(x, vals, bottom=bottom, color=color, label=label)                                     # draw this PC layer on top of the previous bottom
    bottom = bottom + vals                                                                        # advance bottom for the next iteration
ax.set_xticks(list(x))                                                                            # explicit tick positions
ax.set_xticklabels(drill_df['group'], rotation=60, ha='right', fontsize=8)                       # group labels at 60deg right-aligned; small font so they fit
ax.set_ylabel('Variance explained within group')
ax.set_title('Per-eLife-group drill-down: variance across top 3 within-group PCs')
ax.axhline(0.9, color='black', linestyle='--', linewidth=0.6, alpha=0.5)                         # reference line at 90 percent: groups whose top-3 stack reaches it are "compact" in 3D
ax.legend()
stamp_version(fig, label='08 drill-down')                                                        # version stamp in the footer for traceability
plt.tight_layout()                                                                                # tighten margins so labels don't clip
plt.savefig(EXAMPLE_OUTPUT_DIR / '08_drill_down.png', dpi=150, bbox_inches='tight')              # write committed example PNG; dpi=150 for crisp output
plt.show()


## 3. Nested LMM comparison

Fit a sequence of LMMs on the chosen target kinematic feature,
adding connectivity-region covariates in order of prior
priority. Compare via AIC / BIC / LRT. A small ``p_vs_prior``
means the step added predictive power beyond the previous model.

At small N the LMMs are underdetermined whenever there are more
covariates than subjects; synthetic mode (N=30) is the right
test bench for this section.


In [ ]:
reach_level = data.AKDdf[data.AKDdf['contact_group'] == 'contacted'].copy()                       # work on per-reach rows; restrict to reaches that actually contacted the pellet (kinematics need a contact event)
reach_level['phase_group'] = pd.Categorical(                                                      # convert phase to ordered categorical so statsmodels treats it as a factor with intentional ordering
    reach_level['phase_group'], categories=list(ANALYZABLE_PHASES), ordered=True,                  # category order taken from the parameters cell list (Pre, post-injury, post-rehab, etc.)
)

# Attach top-K prior-ranked connectivity region values to every reach
# by merging on subject_id from the wide connectivity matrix.
canonical_cols = ordered_hemisphere_columns(                                                       # helper returns columns in the prior-defined region order, restricted to those actually present
    SKILLED_REACHING, available=data.FCDGdf_wide.columns.tolist(),                                 # SKILLED_REACHING: the prior list (CST > RuST > ReST etc.); available filters out missing columns
)
top_cols = [c for c in canonical_cols if c.endswith('_both')][:TOP_K_PRIORS]                       # keep bilateral '_both' versions only (avoids double-counting hemispheres) and slice the top-K from the parameters cell
conn_wide = data.FCDGdf_wide[top_cols].fillna(0)                                                   # subset wide matrix to the chosen columns; fill missing with 0 since absent data ~= no traced cells
# Rename to safe python identifiers for patsy formula
safe_names = {c: f"conn_{i}" for i, c in enumerate(top_cols)}                                      # patsy chokes on column names containing dots/dashes; map to conn_0, conn_1, ...
conn_wide = conn_wide.rename(columns=safe_names).reset_index()                                     # apply rename and lift subject_id from the index back to a column for the merge
reach_level = reach_level.merge(conn_wide, on='subject_id', how='left')                            # left-join: every reach now carries its subject's connectivity values; subjects missing connectivity get NaN

rhs_parts = list(safe_names.values())                                                              # list of patsy-safe predictor names (the conn_N columns)
model_specs = [                                                                                    # nested model sequence: each spec is (label, additional_RHS_terms)
    ('baseline (phase only)', ''),                                                                 # baseline: phase fixed effect only (added by helper); empty string means no extra covariates
    (f'+top{TOP_K_PRIORS}_priors', ' + '.join(rhs_parts)),                                         # extended: baseline + all top-K connectivity regions joined with patsy '+'
]

nested_results = compare_nested_lmms(                                                              # helper fits each model in sequence and returns AIC, BIC, log-likelihood, LRT p-value vs previous spec
    reach_level.dropna(subset=rhs_parts + [TARGET_FEATURE]),                                       # drop rows missing any predictor or the target so all models compare on the same data
    target_feature=TARGET_FEATURE,                                                                 # name of kinematic feature to model (from parameters cell)
    model_specs=model_specs,                                                                       # the nested sequence defined above
    groups='subject_id',                                                                           # outermost random-effect grouping: subject
    vc_formula={'session': '0 + C(session_date)'},                                                 # nested random effect: session_date within subject; '0 +' suppresses intercept so each session gets its own random slope
)
print(nested_results.to_string(index=False))                                                        # tabular AIC/BIC/LRT summary; small p_vs_prior column means the added regions matter


## 4. Prior-weighted PCA

Re-run the connectivity PCA after multiplying each z-scored
feature by ``exp(-PRIOR_DECAY * rank)`` where ``rank`` is the
region's position in the prior. This amplifies variance from
priority regions; PC1 is then forced to preferentially load on
them unless a low-priority region truly dominates.

Compare the weighted and unweighted top loadings side by side.
Regions that appear in both are robust; those that appear only
unweighted are data-driven additions the prior de-emphasized.


In [ ]:
X_conn = data.FCDGdf_wide.fillna(0)                                                               # wide subject x region matrix; fill NaNs with 0 so PCA has no missing values
X_ordered = X_conn[canonical_cols]                                                                # reorder columns into prior-priority order so weights align by position later
X_scaled = pd.DataFrame(                                                                          # z-score each column (mean 0, std 1) so PCA isn't dominated by raw-count magnitude
    StandardScaler().fit_transform(X_ordered),                                                    # sklearn StandardScaler: subtracts column mean, divides by std
    columns=X_ordered.columns, index=X_ordered.index,                                              # rebuild a dataframe so column/index labels survive the scaler call
)

weights = priority_weights_from_prior(SKILLED_REACHING, X_ordered.columns, decay=PRIOR_DECAY)     # helper builds w_i = exp(-decay * rank_i); higher decay = more aggressive down-weighting of low-priority regions
X_weighted = apply_feature_weights(X_scaled, weights)                                              # element-wise multiply each column by its weight; PCA on this is the prior-weighted version

pca_raw = PCA(n_components=min(3, len(X_ordered) - 1)).fit(X_scaled.values)                       # sklearn PCA on raw z-scored matrix; n_components capped at min(3, N-1) since PCA can't exceed N-1 components
pca_weighted = PCA(n_components=min(3, len(X_ordered) - 1)).fit(X_weighted.values)                # PCA on weighted matrix; same cap so the two are comparable

loadings_raw = pd.Series(pca_raw.components_[0], index=X_scaled.columns)                          # PC1 loadings (eigenvector); components_[0] is the first principal component
loadings_weighted = pd.Series(pca_weighted.components_[0], index=X_scaled.columns)                # weighted PC1 loadings; same shape, different values

top_raw = loadings_raw.abs().nlargest(10).index                                                   # 10 regions with largest absolute loading on raw PC1 (sign-agnostic since PC sign is arbitrary)
top_weighted = loadings_weighted.abs().nlargest(10).index                                         # same for weighted PC1

print('Top 10 regions on PC1 (unweighted):')
print(loadings_raw.loc[top_raw].sort_values())                                                    # sort by signed value so reader sees positive vs negative loadings clearly
print()
print('Top 10 regions on PC1 (prior-weighted):')
print(loadings_weighted.loc[top_weighted].sort_values())

# Side-by-side horizontal bar chart of top 15 regions by |loading|, both versions
union = list(pd.Index(top_raw).union(pd.Index(top_weighted))[:15])                                # union of top-10 from each version, truncated to 15 to keep the chart readable
comp = pd.DataFrame({                                                                             # build a 2-column comparison dataframe (rows=regions, columns=unweighted/prior-weighted)
    'unweighted': loadings_raw.loc[union],
    'prior-weighted': loadings_weighted.loc[union],
}).reindex(union)                                                                                 # ensure row order matches the 'union' list (avoids alphabetical reshuffling)
fig, ax = plt.subplots(figsize=FIGSIZE_WEIGHTED)                                                  # figsize from parameters cell
comp.plot(kind='barh', ax=ax, width=0.8)                                                          # horizontal bars; pandas plots the two columns side-by-side per region
ax.axvline(0, color='black', linewidth=0.5)                                                       # zero reference line so positive vs negative loadings are visually obvious
ax.set_xlabel('PC1 loading')
ax.set_title(f'PC1 loadings: unweighted vs prior-weighted (decay={PRIOR_DECAY})')                 # title includes the decay value so readers know which weighting regime they're seeing
ax.invert_yaxis()                                                                                 # put highest-priority region at the top (matplotlib default puts row-0 at the bottom for barh)
stamp_version(fig, label='08 prior weighted')                                                     # version stamp in the footer
plt.tight_layout()                                                                                 # margin tightening
plt.savefig(EXAMPLE_OUTPUT_DIR / '08_prior_weighted.png', dpi=150, bbox_inches='tight')           # write committed example PNG
plt.show()


## Summary

Together, the four tools above test whether the canonical
pipeline's implicit choices -- eLife grouping for decomposition,
unweighted PCA, phase-only LMMs -- lose information that prior
knowledge or finer-grained data could recover. Interpretation
guide:

- If grouped and ungrouped scree plots match closely, the eLife
  grouping isn't discarding much. If they diverge, the atomic
  analysis deserves a seat at the table.
- If within-group drill-downs show one group with distributed
  variance across PCs, that group has subregional heterogeneity
  the paper should call out.
- If the nested LMM's ``p_vs_prior`` is small, the added
  regions carry real independent information; the prior alone
  wasn't enough.
- If prior-weighted PC1 differs from unweighted PC1, the
  ordering matters and should be discussed. If they match,
  the dominant variance axes were already aligned with the
  prior.

Paper-readiness roadmap (tensor decomposition, cross-validated
prediction, supervised clustering, session-level growth curves,
outlier-aware modeling) is tracked in
[`project_analysis_roadmap`](../docs/assumptions.md) and in the
user memory. Revisit when N scales.


<!-- INTERPRETATION_BLOCK -->
## How to read this notebook's output

<details>
<summary>What the four hypothesis-informed tests tell you (click to expand)</summary>

**Grouped vs ungrouped PCA scree comparison**: does the eLife grouping
hide variance?

- Grouped and ungrouped curves track each other closely = eLife
  grouping captures the dominant variance structure; aggregating to
  group level loses little.
- Ungrouped PC1+PC2 explain notably MORE than the grouped equivalent
  = atomic-region variance is being averaged out by the grouping;
  drill-down (next panel) is justified.
- Ungrouped PC1+PC2 explain notably LESS = the grouping is creating
  structure that doesn't exist at the atomic level (rare but
  possible).

**Per-group drill-down stacked bar**: each bar is one eLife group, its
height shows variance explained by the top 3 within-group PCs.

- Bars dominated by blue (PC1) = group is internally coherent; one
  axis captures all subregional variation; the eLife grouping is
  defensible for that group.
- Bars where PC1, PC2, PC3 are all substantial = group has subregional
  heterogeneity that the eLife grouping is hiding. These groups are
  candidates for "we should split this into subregions in future
  work."
- Dashed line at 0.9 = visual reference for "PC1+PC2+PC3 capture
  almost everything"; bars below the line indicate even more variance
  is in higher-order PCs.

**Nested LMM comparison table**:

- AIC and BIC: smaller = better. If `+top5_priors` has a smaller AIC
  than `baseline (phase only)`, adding the top-5 prior-ranked regions
  improves the model fit beyond what phase alone explains.
- `p_vs_prior`: the LRT p-value for the full model vs the previous
  model. Small (<0.05) = the added connectivity regions explain
  variance the prior model couldn't. Large = the simpler model was
  sufficient.
- At small N with reach-level data the LMM has plenty of within-subject
  data points but only 4 inferential units; LRT is anti-conservative.
  Synthetic mode (USE_SYNTHETIC=True) is the right test bench.

**Prior-weighted PCA comparison bar chart**:

- Top regions on PC1 (unweighted) and (prior-weighted) overlap heavily
  = the dominant variance axis is already aligned with the prior;
  the prior didn't change what PC1 sees.
- Lists differ substantially = the unweighted PCA was driven by
  regions the prior de-emphasizes. Worth investigating: are those
  regions truly unimportant for reaching (literature might be missing
  something) or noisy / artifactual?

The four tests collectively answer: should we trust the eLife grouping
and the literature-derived priors, or does the data argue for a more
data-driven feature selection?

</details>
